# Phase 1 runner — long-context single-request sweep

Runs the three Phase-1 models (Llama-3.1-8B-Instruct, Qwen3-8B, Qwen2-VL-7B-Instruct) at 8k/16k/32k/64k context, batch size 1, on the available GPU. OOM is recorded as `success: false` rather than crashing the sweep.

**Before running:** the repo must already be at `/content/LLM_Inference` (clone it manually if it isn't). Run cells top-to-bottom. Cell 4 is a 1024-token dry-run that catches wiring problems before the long runs; **inspect Cell 5's output** before kicking off Cell 6.


In [ ]:
# 1. Setup — install deps, point cwd at the repo.
import os
os.chdir('/content/LLM_Inference')
!pip install -q -r requirements.txt
!pip install -q bitsandbytes  # required only for the 4-bit T4 extension cell at the end
print('cwd:', os.getcwd())


In [ ]:
# 2. Hugging Face login (Llama-3.1 is gated; the Qwens are not).
from huggingface_hub import login
login()


In [ ]:
# 3. Pull the latest backend code into this Python process.
#    The local files have the YaRN+rope_parameters merge fix; we just need to
#    drop any stale cached imports from earlier in the session.
import sys, importlib
for m in [k for k in sys.modules if k.startswith(('src.backends', 'src.benchmark'))]:
    del sys.modules[m]
sys.path.insert(0, '/content/LLM_Inference')
from src.backends.hf_backend import HFBackend, _normalize_rope
existing = {'rope_type': 'default', 'base': 1_000_000.0}
merged = _normalize_rope({'type': 'yarn', 'factor': 2.0, 'original_max_position_embeddings': 32768}, existing)
assert merged['base'] == 1_000_000.0, 'rope merge dropped base — fix not in effect'
assert merged['rope_type'] == 'yarn'
print('rope merge sanity check passed:', merged)


In [ ]:
# 4. Dry-run all three models at context_length=1024, max_new=32.
#    ~3-5 minutes total on A100. Catches schema/version drift before the long
#    sweep. Order: Qwen3 first (32k native), VLM next, Llama-3.1 last.
!python3 scripts/run_context_sweep.py --config configs/baseline_hf.yaml \
  --model-config configs/qwen3_8b.yaml \
  --context-lengths 1024 --max-new-tokens 32 \
  --results-path results/dryrun.jsonl

!python3 scripts/run_vlm_context_sweep.py --config configs/baseline_hf.yaml \
  --model-config configs/qwen2_vl_7b.yaml \
  --context-lengths 1024 --max-new-tokens 32 \
  --results-path results/dryrun.jsonl

!python3 scripts/run_context_sweep.py --config configs/baseline_hf.yaml \
  --model-config configs/llama3_1_8b_instruct.yaml \
  --context-lengths 1024 --max-new-tokens 32 \
  --results-path results/dryrun.jsonl


In [ ]:
# 5. Inspect the dry-run rows. All three should show:
#      - success=True
#      - ttft < total_latency
#      - ttft/total_latency in roughly the 0.3-0.7 range
#    If any row fails or TTFT looks wrong, STOP and debug before Cell 6.
import json
print(f"{'model':35} {'success':>8} {'ttft':>8} {'total':>8} {'ratio':>6} {'peak_gb':>8}")
print('-' * 80)
for line in open('results/dryrun.jsonl'):
    r = json.loads(line)
    name = r['model_name'][:34]
    if r['success']:
        ratio = r['ttft_seconds'] / r['total_latency_seconds']
        peak = r['peak_gpu_memory_gb'] or 0
        print(f"{name:35} {'OK':>8} {r['ttft_seconds']:8.3f} {r['total_latency_seconds']:8.3f} {ratio:6.2f} {peak:8.2f}")
    else:
        print(f"{name:35} {'FAIL':>8}   error: {r['error'][:60]}")


## Full sweep — only run after Cell 5 looks good

Each command writes one JSONL file per `<model, hardware>` so cross-platform results merge cleanly later. Set `HARDWARE` below to either `a100` or `t4` so the filenames stay sortable.


In [ ]:
HARDWARE = 'a100'   # change to 't4' when running on the T4 box
print(f'Will write results to: results/phase1_<model>_{HARDWARE}.jsonl')


In [ ]:
# 6a. Llama-3.1-8B-Instruct sweep.
!python3 scripts/run_context_sweep.py --config configs/baseline_hf.yaml \
  --model-config configs/llama3_1_8b_instruct.yaml \
  --context-lengths 8192 16384 32768 65536 \
  --max-new-tokens 128 \
  --results-path results/phase1_llama31_$HARDWARE.jsonl


In [ ]:
# 6b. Qwen3-8B (reasoning architecture) sweep — YaRN factor=2 active.
!python3 scripts/run_context_sweep.py --config configs/baseline_hf.yaml \
  --model-config configs/qwen3_8b.yaml \
  --context-lengths 8192 16384 32768 65536 \
  --max-new-tokens 128 \
  --results-path results/phase1_qwen3_$HARDWARE.jsonl


In [ ]:
# 6c. Qwen2-VL-7B-Instruct (vision-language) sweep — YaRN factor=2 active,
#     image-token count subtracted from text target so total context == target.
!python3 scripts/run_vlm_context_sweep.py --config configs/baseline_hf.yaml \
  --model-config configs/qwen2_vl_7b.yaml \
  --context-lengths 8192 16384 32768 65536 \
  --max-new-tokens 128 \
  --results-path results/phase1_qwen2vl_$HARDWARE.jsonl


## Extension experiment — T4 only

Run only on the T4 box. NF4 4-bit quantization via bitsandbytes; same sweep, same metrics, separate JSONL so it can be compared 1:1 against `phase1_qwen3_t4.jsonl`.


In [ ]:
# 7. Qwen3-8B 4-bit (NF4) extension on T4.
!python3 scripts/run_context_sweep.py --config configs/baseline_hf.yaml \
  --model-config configs/qwen3_8b_4bit_t4.yaml \
  --context-lengths 8192 16384 32768 65536 \
  --max-new-tokens 128 \
  --results-path results/phase1_qwen3_4bit_t4.jsonl


In [ ]:
# 8. Aggregate everything in results/ into a summary table.
!python3 scripts/summarize_results.py --results-dir results/
